# 03b — SigLIP 2 Dual-Head ViT (Linear Probe & Fine-Tuning)

Trains and evaluates **`google/siglip2-base-patch16-384`** (`ViT-B/16@384px`) on the Timeguessr AI task using the **`semantic_medium`** discretization (`1,249` country-pure tiles) and exact scoring formulas from `formulas.txt`.

**Two-Stage Pipeline (identical recipe to `04a_streetclip.ipynb`):**
1. **Stage 1 — Linear Probe (`outputs/models/siglip2_linear_probe/`):** Extract & cache frozen SigLIP 2 embeddings once, then train dual MLP heads with Gaussian year smoothing (`sigma=2.0`) and GPU geodesic distance soft labels (`tau=75 km`).
2. **Stage 2 — Fine-Tuning (`outputs/models/siglip2_finetune/`):** Warm-start the dual heads from Stage 1, unfreeze the **last 6 ViT blocks**, and fine-tune end-to-end in `bfloat16` with gradient checkpointing and geolocation-safe augmentations (no horizontal flip).

## 1. Setup & Configuration

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from timeguessr_ai import (
    clear_device_cache,
    extract_and_cache_embeddings,
    format_device_info,
    get_default_device,
    load_cached_embeddings,
    load_dataset_and_tiles,
    train_finetune,
    train_linear_probe,
)

DEVICE = str(get_default_device())
print(f"Project root: {PROJECT_ROOT}")
print(f"Device:       {format_device_info(DEVICE)}")

W1124 12:37:37.674000 19624 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Project root: d:\VSCodeProjects\CS441
Device:       cuda (NVIDIA GeForce RTX 4070 SUPER)


In [3]:
CFG = {
    "model_id": "google/siglip2-base-patch16-384",
    "model_label": "SigLIP 2 (ViT-B/16@384)",
    "tile_scheme": "semantic_medium",
    "sigma_year": 2.0,
    "tau_geo_km": 135.0,
    "country_alpha": 0.7,
    "w_time": 1.0,
    "dropout": 0.2,
    "seed": 42,
    # Stage 1: Embedding extraction & Linear Probe
    "use_cached_embeddings": False,  # Load & sanity-check cached .npy embeddings; False = re-extract
    "extract_batch_size": 64,
    "lp_epochs": 25,
    "lp_batch_size": 512,
    "lp_lr": 1e-3,
    # Stage 2: Fine-tuning (tuned for 12 GB VRAM RTX 4070 SUPER)
    "unfreeze_last_n": 6,
    "ft_epochs": 5,
    "ft_batch_size": 32,
    "ft_grad_accum": 2,  # effective batch size = 64 (matches StreetCLIP)
    "ft_backbone_lr": 2e-5,
    "ft_head_lr": 3e-4,
    "num_workers": 0,    # 0 is safest on Windows Jupyter; increase to 2-4 on Linux
}

LP_OUT_DIR = PROJECT_ROOT / "outputs" / "models" / "siglip2_linear_probe"
FT_OUT_DIR = PROJECT_ROOT / "outputs" / "models" / "siglip2_finetune"

splits, cells_df, tile_col = load_dataset_and_tiles(PROJECT_ROOT, tile_scheme=CFG["tile_scheme"])
print(f"Tiles ({CFG['tile_scheme']}): {len(cells_df):,}")
for k, v in splits.items():
    print(f"  {k:16s}: {len(v):7,d} images")

Tiles (semantic_medium): 841
  train           :  87,056 images
  val             :   9,584 images
  test_timeguessr :   1,600 images


## 2. Stage 1 — Extract Frozen Embeddings & Train Linear Probe

Embeddings are cached to `data/embeddings/google__siglip2-base-patch16-384/` so Stage 1 runs in seconds and Notebook 02 (FAISS KNN) can reuse them directly.

In [4]:
if CFG["use_cached_embeddings"]:
    embeddings = load_cached_embeddings(
        model_id=CFG["model_id"],
        splits=splits,
        project_root=PROJECT_ROOT,
    )
else:
    embeddings = extract_and_cache_embeddings(
        model_id=CFG["model_id"],
        splits=splits,
        project_root=PROJECT_ROOT,
        batch_size=CFG["extract_batch_size"],
        num_workers=CFG["num_workers"],
        device=DEVICE,
        force=True,
    )

Extracting embeddings for google/siglip2-base-patch16-384 on cuda (torch.bfloat16) for splits: ['train', 'val', 'test_timeguessr']


Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

[transformers] SiglipVisionModel LOAD REPORT from: google/siglip2-base-patch16-384
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.head.bias                 

Extract train:   0%|          | 0/1361 [00:00<?, ?it/s]

d:\VSCodeProjects\CS441\.venv\Lib\site-packages\PIL\Image.py:3593: DecompressionBombWarning: Image size (89822655 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Saved train: shape=(87056, 768), size=133.7 MB -> d:\VSCodeProjects\CS441\data\embeddings\google__siglip2-base-patch16-384\train_embeddings.npy


Extract val:   0%|          | 0/150 [00:00<?, ?it/s]

Saved val: shape=(9584, 768), size=14.7 MB -> d:\VSCodeProjects\CS441\data\embeddings\google__siglip2-base-patch16-384\val_embeddings.npy


Extract test_timeguessr:   0%|          | 0/25 [00:00<?, ?it/s]

Saved test_timeguessr: shape=(1600, 768), size=2.5 MB -> d:\VSCodeProjects\CS441\data\embeddings\google__siglip2-base-patch16-384\test_timeguessr_embeddings.npy


In [5]:
lp_result = train_linear_probe(
    embeddings=embeddings,
    splits=splits,
    cells_df=cells_df,
    output_dir=LP_OUT_DIR,
    run_config={**CFG, "stage": "linear_probe", "run_name": "siglip2_linear_probe"},
    epochs=CFG["lp_epochs"],
    batch_size=CFG["lp_batch_size"],
    lr=CFG["lp_lr"],
    sigma_year=CFG["sigma_year"],
    tau_geo_km=CFG["tau_geo_km"],
    w_time=CFG["w_time"],
    dropout=CFG["dropout"],
    country_alpha=CFG["country_alpha"],
    device=DEVICE,
    seed=CFG["seed"],
)

print("\n=== Stage 1 (SigLIP 2 Linear Probe) — Held-Out Timeguessr Test ===")
display(pd.DataFrame([lp_result["test_metrics"]]).T.rename(columns={0: "SigLIP 2 Linear Probe"}))

d:\VSCodeProjects\CS441\src\timeguessr_ai\data.py:218: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  self.year_idx = torch.from_numpy(df["year_idx"].to_numpy(dtype=np.int64))


Epoch 01/25 ( 2.1s) | loss=7.2064 (s=4.132, t=3.075) | val_total=7345.0 (loc=3134.6, yr=4210.5) | med_km=470.3 | yr_mae=5.20
Epoch 02/25 ( 2.1s) | loss=6.3390 (s=3.477, t=2.862) | val_total=7509.5 (loc=3242.1, yr=4267.4) | med_km=393.9 | yr_mae=4.86
Epoch 03/25 ( 2.2s) | loss=6.0647 (s=3.275, t=2.790) | val_total=7544.2 (loc=3290.6, yr=4253.6) | med_km=359.3 | yr_mae=4.92
Epoch 04/25 ( 2.4s) | loss=5.8718 (s=3.140, t=2.732) | val_total=7574.2 (loc=3259.0, yr=4315.3) | med_km=364.4 | yr_mae=4.59
Epoch 05/25 ( 2.3s) | loss=5.7110 (s=3.019, t=2.692) | val_total=7625.1 (loc=3310.3, yr=4314.9) | med_km=336.8 | yr_mae=4.61
Epoch 06/25 ( 2.1s) | loss=5.5784 (s=2.928, t=2.650) | val_total=7659.1 (loc=3314.9, yr=4344.2) | med_km=328.1 | yr_mae=4.40
Epoch 07/25 ( 2.1s) | loss=5.4565 (s=2.838, t=2.619) | val_total=7698.6 (loc=3342.2, yr=4356.4) | med_km=325.0 | yr_mae=4.35
Epoch 08/25 ( 2.1s) | loss=5.3684 (s=2.778, t=2.591) | val_total=7712.9 (loc=3357.2, yr=4355.7) | med_km=319.6 | yr_mae=4.34


,SigLIP 2 Linear Probe
n_samples,1600.00
total_score_mean,6565.50
total_score_median,6864.62
loc_score_mean,2867.08
loc_score_median,3464.44
year_score_mean,3698.42
year_score_median,4051.14
dist_err_median_km,679.49
dist_err_mean_km,2152.36
within_25km_pct,0.06


## 3. Stage 2 — Fine-Tune Last 6 ViT Blocks (Warm-Started from Stage 1)

Loads the best dual MLP heads from Stage 1, unfreezes the final 6 transformer blocks (+ MAP attention pooler) of SigLIP 2, and fine-tunes end-to-end with differential learning rates (`2e-5` backbone / `3e-4` heads), `bfloat16`, and gradient checkpointing.

In [ ]:
# Free cached embeddings from RAM before Stage 2 image fine-tuning
if "embeddings" in globals():
    del embeddings
clear_device_cache(DEVICE)

ft_result = train_finetune(
    model_id=CFG["model_id"],
    splits=splits,
    cells_df=cells_df,
    project_root=PROJECT_ROOT,
    output_dir=FT_OUT_DIR,
    run_config={**CFG, "stage": "finetune", "run_name": "siglip2_finetune"},
    warmstart_checkpoint=LP_OUT_DIR / "best_checkpoint.pt",
    unfreeze_last_n=CFG["unfreeze_last_n"],
    epochs=CFG["ft_epochs"],
    batch_size=CFG["ft_batch_size"],
    grad_accum_steps=CFG["ft_grad_accum"],
    backbone_lr=CFG["ft_backbone_lr"],
    head_lr=CFG["ft_head_lr"],
    sigma_year=CFG["sigma_year"],
    tau_geo_km=CFG["tau_geo_km"],
    w_time=CFG["w_time"],
    dropout=CFG["dropout"],
    country_alpha=CFG["country_alpha"],
    num_workers=CFG["num_workers"],
    device=DEVICE,
    seed=CFG["seed"],
    resume=True,
)

print("\n=== Stage 2 (SigLIP 2 Fine-Tune) — Held-Out Timeguessr Test ===")
display(pd.DataFrame([ft_result["test_metrics"]]).T.rename(columns={0: "SigLIP 2 Fine-Tune"}))